In [5]:
import json
import sys

import httpx

API_URL = "http://localhost:7100"

In [6]:
query = "medical graph rag"
payload = {
    "query": query,
    "num_topics": 1, # fan-out only 1 for visibility
    "max_attempts": 2,
    "top_k": 5,
}

In [7]:
def handle_event(event_type: str | None, data: dict) -> None:
    if event_type == "node_start":
        node = data.get("node", "?")
        topic = data.get("topic")
        label = f"  [{topic}]" if topic else ""
        print(f"▶ {node}{label}")

    elif event_type == "node_end":
        node = data.get("node", "?")
        topic = data.get("topic")
        detail = _end_detail(node, data)
        label = f"  [{topic}]" if topic else ""
        print(f"✓ {node}{label}  {detail}")

    elif event_type == "done":
        topics = data.get("topics", [])
        papers = data.get("papers", [])
        print()
        print(f"══ Done ══  topics={topics}")
        for i, p in enumerate(papers, 1):
            score = p.get("relevance_score", 0)
            title = p.get("title", "Untitled")
            print(f"  {i}. [{score:.2f}] {title}")

    elif event_type == "error":
        print(f"✗ ERROR: {data.get('error', data)}")

    else:
        print(f"  ({event_type}) {json.dumps(data)[:120]}")


def _end_detail(node: str, data: dict) -> str:
    if node == "generate_topics":
        return f"topics={data.get('topics', [])}"
    if node == "generate_query":
        return f"query={data.get('search_query', '')!r}"
    if node == "search":
        return f"papers_found={data.get('papers_found', 0)}"
    if node == "judge":
        return f"papers_relevant={data.get('papers_relevant', 0)}"
    if node == "generate_feedback":
        fb = (data.get("feedback") or "")[:80]
        return f"feedback={fb!r}"
    if node == "collect_and_rank":
        return f"papers={data.get('papers_count', 0)}"
    if node == "search_subgraph":
        return f"emitted={data.get('papers_emitted', 0)}"
    return ""

In [8]:
# Stream
with httpx.stream(
    "POST",
    f"{API_URL}/search/stream",
    json=payload,
    timeout=120.0,
) as resp:
    resp.raise_for_status()

    event_type = None
    for line in resp.iter_lines():
        print(line)
        if line.startswith("event: "):
            event_type = line[7:].strip()
        elif line.startswith("data: "):
            data = json.loads(line[6:])
            handle_event(event_type, data)
        elif line == "":
            event_type = None

event: node_start
data: {"node": "generate_topics"}
▶ generate_topics

event: node_end
data: {"node": "generate_topics", "topics": ["\uc758\ub8cc \uadf8\ub798\ud504 RAG \uc2dc\uc2a4\ud15c\uc758 \uc815\ud655\ub3c4 \ubc0f \uc2e0\ub8b0\uc131 \ud5a5\uc0c1\uc744 \uc704\ud55c \ub3c4\uba54\uc778 \ud2b9\ud654 \uc9c0\uc2dd \uadf8\ub798\ud504 \uad6c\ucd95 \uc804\ub7b5", "\uc758\ub8cc RAG \uc2dc\uc2a4\ud15c\uc758 \uc0ac\uc801 \uc815\ubcf4 \ubcf4\ud638 \ubc0f \ub370\uc774\ud130 \ud504\ub77c\uc774\ubc84\uc2dc\ub97c \uc704\ud55c \uc811\uadfc \uc81c\uc5b4 \ubc0f \uc554\ud638\ud654 \uae30\ubc95"]}
✓ generate_topics  topics=['의료 그래프 RAG 시스템의 정확도 및 신뢰성 향상을 위한 도메인 특화 지식 그래프 구축 전략', '의료 RAG 시스템의 사적 정보 보호 및 데이터 프라이버시를 위한 접근 제어 및 암호화 기법']

event: node_start
data: {"node": "search_subgraph", "topic": "\uc758\ub8cc \uadf8\ub798\ud504 RAG \uc2dc\uc2a4\ud15c\uc758 \uc815\ud655\ub3c4 \ubc0f \uc2e0\ub8b0\uc131 \ud5a5\uc0c1\uc744 \uc704\ud55c \ub3c4\uba54\uc778 \ud2b9\ud654 \uc9c0\uc2dd \uadf8\ub798\ud504 \uad6c\u